In [3]:
import pandas as pd
import os
import shutil
import numpy as np
from collections import Counter
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
from IPython.display import FileLink,display
import gradio as gr

from sklearn.metrics import classification_report ,confusion_matrix,accuracy_score,precision_score,recall_score,f1_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import callbacks,losses,layers,optimizers
from tensorflow.keras.layers import Dropout,Dense,Conv2D,MaxPooling2D,BatchNormalization,Input,Flatten,ReLU
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.applications import MobileNetV3Small


In [5]:
original_dirs={
    'Calculus':'/kaggle/input/datasets/salmansajid05/oral-diseases/Calculus/Calculus',
    'Caries':'/kaggle/input/datasets/salmansajid05/oral-diseases/Data caries/Data caries/caries augmented data set/preview',
    'Gingivitis':'/kaggle/input/datasets/salmansajid05/oral-diseases/Gingivitis/Gingivitis',
    'Ulcers':'/kaggle/input/datasets/salmansajid05/oral-diseases/Mouth Ulcer/Mouth Ulcer/Mouth_Ulcer_augmented_DataSet/preview',
    'Tooth Dicoloration':'/kaggle/input/datasets/salmansajid05/oral-diseases/Tooth Discoloration/Tooth Discoloration /Tooth_discoloration_augmented_dataser/preview',
    'Hypodontia':'/kaggle/input/datasets/salmansajid05/oral-diseases/hypodontia/hypodontia'
}

In [6]:
original_dirs.keys()

dict_keys(['Calculus', 'Caries', 'Gingivitis', 'Ulcers', 'Tooth Dicoloration', 'Hypodontia'])

In [7]:
base_dir='/kaggle/working/'
splits=['train','val','test']
classes=list(original_dirs.keys())

In [51]:
classes

['Calculus',
 'Caries',
 'Gingivitis',
 'Ulcers',
 'Tooth Dicoloration',
 'Hypodontia']

In [8]:
for split in splits:
    for class_name in classes:
        os.makedirs(os.path.join(base_dir,split,class_name),exist_ok=True)

In [9]:
def copy_and_count_images(class_name,image_paths):
    train_paths,test_paths=train_test_split(image_paths,test_size=0.1,random_state=22)
    train_paths,val_paths=train_test_split(train_paths,test_size=0.2,random_state=22)
    split_paths={'train':train_paths,'val':val_paths,'test':test_paths}
    for split ,paths in split_paths.items():
        for img_path in paths:
            target_path=os.path.join(base_dir,split,class_name,os.path.basename(img_path))
            shutil.copy(img_path,target_path)

In [29]:
for class_name,original_dir in original_dirs.items():
    image_paths=[os.path.join(root,file)for root,_,files in os.walk(original_dir)
                for file in files if file.endswith(('.jpg','.jpeg','.png'))]
    if image_paths:
        copy_and_count_images(class_name,image_paths)

In [ ]:
TrainPath='/kaggle/working/train'
TestPath='/kaggle/working/test'
ValidPath='/kaggle/working/val'
batchsize=32
image_height=256
image_width=256
epochs=100
num_classes=6


In [ ]:
train_ds=image_dataset_from_directory(
    TrainPath,
    labels='inferred',
    batch_size=batchsize,
    image_size=(image_width,image_height),
    seed=0,
    shuffle=True,
    class_names=classes
    
)
valid_ds=image_dataset_from_directory(
    ValidPath,
    labels='inferred',
    batch_size=batchsize,
    image_size=(image_width,image_height),
    seed=0,
    shuffle=True,
    class_names=classes
    
)
test_ds=image_dataset_from_directory(
    TestPath,
    labels='inferred',
    batch_size=batchsize,
    image_size=(image_width,image_height),
    seed=0,
    shuffle=True,
    class_names=classes
    
)

In [ ]:
for image ,labels in train_ds.take(1):
    plt.figure(figsize=(10,10))
    for i in range(9):
        ax=plt.subplot(3,3,i+1)
        plt.imshow(image[i].numpy().astype('uint8'))
        plt.title(f'label:{labels[i].numpy()}')
        plt.axis('off')
    plt.show()
    

In [58]:
len(train_ds)

242

In [ ]:
all_labels=[]

for image,labels in train_ds:
    all_labels.extend(labels.numpy())
class_counts=Counter(all_labels)
classe=sorted(class_counts.keys())
class_counts=[class_counts[c] for c in classe]
class_name=[c for c in classes]
class_counts

In [ ]:
plt.figure(figsize=(10,10))
sns.barplot(x=class_name,y=class_counts,palette='Blues_d')
plt.show()

In [ ]:
y_train=np.concatenate([y for x,y in train_ds],axis=0)
y_train

In [ ]:
class_weights=compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_wight_dic=dict(enumerate(class_weights))

In [63]:
class_wight_dic

{0: np.float64(1.381080114449213),
 1: np.float64(0.7509723842862699),
 2: np.float64(0.7643507521773555),
 3: np.float64(0.7041393143690736),
 4: np.float64(0.9751262626262627),
 5: np.float64(5.253741496598639)}

In [64]:
for images,labels in train_ds.take(1):
    print(f'image shape:{images.shape}')
    print(f'label:{labels.shape}')
    print(f'labels:{labels}')


image shape:(32, 256, 256, 3)
label:(32,)
labels:[2 2 4 3 4 2 2 4 3 2 5 3 3 3 1 4 3 3 3 3 3 4 4 3 4 2 0 2 1 4 1 1]


In [65]:
train_ds.class_names

['Calculus',
 'Caries',
 'Gingivitis',
 'Ulcers',
 'Tooth Dicoloration',
 'Hypodontia']

In [ ]:
def decode_img(img):
    img=tf.io.decode_image(img,channels=3,expand_animations=False)
    return tf.image.resize(img,[image_height,image_width])
def process_path(filepath):
    img=tf.io.read_file(filepath)
    img=decode_img(img)
    return img
test_ds=tf.data.Dataset.list_files(str(TestPath+"/*/*"),shuffle=False)
test_ds=test_ds.map(process_path,num_parallel_calls=tf.data.AUTOTUNE)

In [ ]:
plt.figure(figsize=(10,10))
i=0
for images in test_ds.take(9):
    plt.subplot(3,3,i+1)
    plt.imshow(images.numpy().astype('uint8'))
    i+=1
    plt.axis('off')
plt.show()  

In [ ]:
img_augmentation=Sequential(
    [
        layers.RandomRotation(factor=0.15),
        layers.RandomTranslation(height_factor=0.1,width_factor=0.3),
        layers.RandomFlip(),
        layers.GaussianNoise(stddev=0.09),
        layers.RandomContrast(factor=0.1)
    ],
    name='img_augmentation'
)

In [ ]:
plt.figure(figsize=(10,10))
for images,labels in train_ds.take(1):
    for i in range(9):
        ax=plt.subplot(3,3,i+1)
        aug_img=img_augmentation(tf.expand_dims(images[i],axis=0))
        plt.imshow(aug_img[0].numpy().astype('uint8'))
        plt.title(classes[labels[i].numpy()])
        plt.axis('off')
        

In [ ]:
AUTOTUNE=tf.data.AUTOTUNE
train_ds=train_ds.cache().prefetch(buffer_size=AUTOTUNE)
validation_ds=valid_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
lr_callback=callbacks.ReduceLROnPlateau(monitor='val_accuracy',factor=0.1,patience=8)
stop_callback=callbacks.EarlyStopping(monitor='val_accuracy',patience=10)
model_ckp=callbacks.ModelCheckpoint('cnn_best_weight.keras',save_best_only=True,monitor='val_accuracy',mode='max')

In [72]:
def cnn_build(input_shape=(image_width,image_height,3),num_classes=len(classes)):
    model=Sequential([
        Input(shape=input_shape),
        img_augmentation,
        layers.Rescaling(1./255),
        Conv2D(32,(3,3),activation='relu',padding='same'),
        MaxPooling2D(pool_size=(2,2)),
        Conv2D(64,(3,3),activation='relu',padding='same'),
        MaxPooling2D(pool_size=(2,2)),
        Conv2D(64,(3,3),activation='relu',padding='same'),
        MaxPooling2D(pool_size=(2,2)),
        Conv2D(128,(3,3),activation='relu',padding='same'),
        MaxPooling2D(pool_size=(2,2)),
        Conv2D(256,(3,3),activation='relu',padding='same'),
        MaxPooling2D(pool_size=(2,2)),
        Flatten(),
        Dense(128,activation='relu'),
        Dropout(0.3),
        Dense(64,activation='relu'),
        Dense(num_classes,activation='softmax')
    ])
    optimizer=Adam(learning_rate=3e-4)
    loss=losses.SparseCategoricalCrossentropy()
    model.compile(optimizer=optimizer,loss=loss,metrics=['accuracy'])
    model.summary()
    return model


In [73]:

cnn=cnn_build()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ img_augmentation (Sequential)   │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling_1 (Rescaling)         │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 256, 256, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 128, 128, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 128, 128, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 64, 64, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 16, 16, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 8, 8, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 16384)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │     2,097,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,531,270 (9.66 MB)

 Trainable params: 2,531,270 (9.66 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history=cnn.fit(
    train_ds,
    validation_data=valid_ds,
    callbacks=[lr_callback,stop_callback,model_ckp],
    verbose=1,
    epochs=epochs,
    class_weight=class_wight_dic
)

Epoch 1/100
242/242 ━━━━━━━━━━━━━━━━━━━━ 22s 79ms/step - accuracy: 0.2459 - loss: 1.6712 - val_accuracy: 0.3488 - val_loss: 1.5368 - learning_rate: 3.0000e-04
Epoch 2/100
242/242 ━━━━━━━━━━━━━━━━━━━━ 17s 70ms/step - accuracy: 0.3628 - loss: 1.4379 - val_accuracy: 0.2946 - val_loss: 1.6156 - learning_rate: 3.0000e-04
Epoch 3/100
242/242 ━━━━━━━━━━━━━━━━━━━━ 17s 69ms/step - accuracy: 0.4172 - loss: 1.2998 - val_accuracy: 0.3948 - val_loss: 1.3044 - learning_rate: 3.0000e-04
Epoch 4/100
242/242 ━━━━━━━━━━━━━━━━━━━━ 16s 68ms/step - accuracy: 0.4550 - loss: 1.2266 - val_accuracy: 0.3948 - val_loss: 1.3130 - learning_rate: 3.0000e-04
Epoch 5/100
242/242 ━━━━━━━━━━━━━━━━━━━━ 17s 69ms/step - accuracy: 0.4705 - loss: 1.1713 - val_accuracy: 0.4966 - val_loss: 1.1178 - learning_rate: 3.0000e-04
Epoch 6/100
242/242 ━━━━━━━━━━━━━━━━━━━━ 17s 70ms/step - accuracy: 0.4910 - loss: 1.1353 - val_accuracy: 0.4160 - val_loss: 1.3391 - learning_rate: 3.0000e-04
Epoch 7/100
242/242 ━━━━━━━━━━━━━━━━━━━━ 17s 7

In [80]:
cnn.save("cnn_best_weight.keras")

In [92]:
display(FileLink("/kaggle/working/cnn_best_weight.keras"))

/kaggle/working/cnn_best_weight.keras

In [93]:
print(os.path.exists("/kaggle/working/cnn_best_weight.keras"))
print(os.path.getsize('/kaggle/working/cnn_best_weight.keras'))

True
30460041


In [85]:
test_ds

<_ParallelMapDataset element_spec=TensorSpec(shape=(256, 256, 3), dtype=tf.float32, name=None)>

In [86]:
print(test_ds.element_spec)

TensorSpec(shape=(256, 256, 3), dtype=tf.float32, name=None)


In [ ]:
# y_test=[]
# y_pred=[]
# for images,labels in test_ds:
#     pred=cnn.predict(images)
#     preds=np.argmax(pred,axis=1)
#     y_pred.extend([preds])
#     y_test.extend(labels.numpy())

In [ ]:
con=confusion_matrix(y_test,y_pred)
plt.figure(figsize=(6,6))
sns.heatmap(con,annot=True,fmt='d',cmap='Blues',yticklabels=classes,xticklabels=classes)
plt.xlabel("Predicted")
plt.ylabel("Acutal")
plt.show()

In [ ]:
model_ckp=callbacks.ModelCheckpoint('efficentNetB4_best_weight.keras',save_best_only=True,monitor='val_accuracy',mode='max')

In [ ]:
def build_model():
    inputs=layers.Input(shape=(image_width,image_height,3))
    x=img_augmentation(inputs)
    model=EfficientNetB4(include_top=False,weights='imagenet',input_tensor=x)
    x=layers.GlobalAveragePooling2D(name='avg_pool')(model.output)
    x=layers.BatchNormalization()(x)
    x=layers.Dropout(0.4)(x)
    output=layers.Dense(num_classes,activation='softmax',name='pred')(x)
    model=tf.keras.Model(inputs,output,name='EfficientNetB4')
    optimizer=optimizers.Adam(learning_rate=3e-4)
    loss=losses.SparseCategoricalCrossentropy()
    model.compile(optimizer=optimizer,loss=loss,metrics=['accuracy'])
    # model.summary()
    return model

In [ ]:
model=build_model()


In [30]:
history=model.fit(
    train_ds,
    validation_data=valid_ds,
    callbacks=[lr_callback,stop_callback,model_ckp],
    verbose=1,
    epochs=epochs
)

Epoch 1/100
242/242 ━━━━━━━━━━━━━━━━━━━━ 283s 1s/step - accuracy: 0.9056 - loss: 0.2400 - val_accuracy: 0.9230 - val_loss: 0.1840 - learning_rate: 3.0000e-04
Epoch 2/100
242/242 ━━━━━━━━━━━━━━━━━━━━ 281s 1s/step - accuracy: 0.9188 - loss: 0.2012 - val_accuracy: 0.9059 - val_loss: 0.2088 - learning_rate: 3.0000e-04
Epoch 3/100
242/242 ━━━━━━━━━━━━━━━━━━━━ 280s 1s/step - accuracy: 0.9284 - loss: 0.1662 - val_accuracy: 0.9204 - val_loss: 0.1617 - learning_rate: 3.0000e-04
Epoch 6/100
242/242 ━━━━━━━━━━━━━━━━━━━━ 282s 1s/step - accuracy: 0.9337 - loss: 0.1531 - val_accuracy: 0.9235 - val_loss: 0.1804 - learning_rate: 3.0000e-04
Epoch 7/100
242/242 ━━━━━━━━━━━━━━━━━━━━ 279s 1s/step - accuracy: 0.9384 - loss: 0.1424 - val_accuracy: 0.9220 - val_loss: 0.1730 - learning_rate: 3.0000e-04
Epoch 8/100
242/242 ━━━━━━━━━━━━━━━━━━━━ 280s 1s/step - accuracy: 0.9298 - loss: 0.1650 - val_accuracy: 0.9173 - val_loss: 0.1662 - learning_rate: 3.0000e-04
Epoch 9/100
242/242 ━━━━━━━━━━━━━━━━━━━━ 280s 1s/ste

In [32]:
model.save("efficentNetB4_best_weight.keras")


In [36]:
model_ckp=callbacks.ModelCheckpoint('mobilenetv3s_best_weight.keras',save_best_only=True,monitor='val_accuracy',mode='max')

In [37]:
def build_model2():
    inputs=layers.Input(shape=(image_width,image_height,3))
    x=img_augmentation(inputs)
    model=MobileNetV3Small(include_top=False,weights='imagenet',input_tensor=x)
    x=layers.GlobalAveragePooling2D(name='avg_pool')(model.output)
    x=layers.BatchNormalization()(x)
    x=layers.Dropout(0.4)(x)
    output=layers.Dense(num_classes,activation='softmax',name='pred')(x)
    model=tf.keras.Model(inputs,output,name='MobileNetV3Small')
    optimizer=optimizers.Adam(learning_rate=3e-4)
    loss=losses.SparseCategoricalCrossentropy()
    model.compile(optimizer=optimizer,loss=loss,metrics=['accuracy'])
    # model.summary()
    return model

In [38]:
model2=build_model2()



4334752/4334752 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [39]:
history=model2.fit(
    train_ds,
    validation_data=valid_ds,
    callbacks=[lr_callback,stop_callback,model_ckp],
    verbose=1,
    epochs=epochs
)

Epoch 1/100
242/242 ━━━━━━━━━━━━━━━━━━━━ 54s 130ms/step - accuracy: 0.6284 - loss: 1.0443 - val_accuracy: 0.7111 - val_loss: 0.8420 - learning_rate: 3.0000e-04
Epoch 2/100
242/242 ━━━━━━━━━━━━━━━━━━━━ 28s 117ms/step - accuracy: 0.7974 - loss: 0.5426 - val_accuracy: 0.8062 - val_loss: 0.4809 - learning_rate: 3.0000e-04
Epoch 3/100
242/242 ━━━━━━━━━━━━━━━━━━━━ 28s 117ms/step - accuracy: 0.8365 - loss: 0.4167 - val_accuracy: 0.8506 - val_loss: 0.4218 - learning_rate: 3.0000e-04
Epoch 4/100
242/242 ━━━━━━━━━━━━━━━━━━━━ 28s 118ms/step - accuracy: 0.8666 - loss: 0.3382 - val_accuracy: 0.8630 - val_loss: 0.3469 - learning_rate: 3.0000e-04
Epoch 5/100
242/242 ━━━━━━━━━━━━━━━━━━━━ 28s 117ms/step - accuracy: 0.8845 - loss: 0.2841 - val_accuracy: 0.8667 - val_loss: 0.3367 - learning_rate: 3.0000e-04
Epoch 6/100
242/242 ━━━━━━━━━━━━━━━━━━━━ 28s 117ms/step - accuracy: 0.8985 - loss: 0.2537 - val_accuracy: 0.8837 - val_loss: 0.3191 - learning_rate: 3.0000e-04
Epoch 7/100
242/242 ━━━━━━━━━━━━━━━━━━━━

In [41]:
model2.save("mobilenetv3s_best_weight.keras")


In [50]:
test_ds=image_dataset_from_directory(
    TestPath,
    labels='inferred',
    batch_size=batchsize,
    image_size=(image_width,image_height),
    seed=0,
    shuffle=True,
    class_names=classes
    
)

Found 1077 files belonging to 6 classes.


In [51]:
models={
    'cnn_model':tf.keras.models.load_model("/kaggle/input/models/shimaagomaa88/cnn-model/keras/default/1/cnn_best_weight.keras"),
    'efficentNet_model':tf.keras.models.load_model("/kaggle/input/models/shimaagomaa88/efficentnet/keras/default/1/efficentNetB4_best_weight.keras"),
    'mobileNet_model':tf.keras.models.load_model("/kaggle/input/models/shimaagomaa88/mobilenet/keras/default/1/mobilenetv3s_best_weight.keras")
}

In [52]:
results={}
for name,model in models.items():
    loss,accuracy=model.evaluate(test_ds)
    results[name]={
        'Loss':loss,
        'Accuracy':accuracy
    }

34/34 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - accuracy: 0.8487 - loss: 0.3852
34/34 ━━━━━━━━━━━━━━━━━━━━ 18s 255ms/step - accuracy: 0.9359 - loss: 0.1453
34/34 ━━━━━━━━━━━━━━━━━━━━ 3s 40ms/step - accuracy: 0.9229 - loss: 0.1840


In [54]:
df=pd.DataFrame(results).T
df=df.sort_values(
    by='Accuracy',
    ascending=False
)
display(df)

,Loss,Accuracy
efficentNet_model,0.145282,0.935933
mobileNet_model,0.183968,0.922934
cnn_model,0.385248,0.848654


In [66]:

def evaluate_model(model, test_ds):

    y_true = []
    y_pred = []

    for images, labels in test_ds:

        predictions = model(images, training=False)

        predicted_classes = np.argmax(
            predictions.numpy(), axis=1
        )

        y_true.extend(labels.numpy())
        y_pred.extend(predicted_classes)

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(
            y_true, y_pred, average='weighted'
        ),
        "Recall": recall_score(
            y_true, y_pred, average='weighted'
        ),
        "F1-score": f1_score(
            y_true, y_pred, average='weighted'
        )
    }

In [71]:
comparison = {}

for name, model in models.items():

    comparison[name] = evaluate_model(model, test_ds)

comparison_df = pd.DataFrame(comparison).T

display(comparison_df.round(4))

,Accuracy,Precision,Recall,F1-score
cnn_model,0.8487,0.8622,0.8487,0.8519
efficentNet_model,0.9359,0.9363,0.9359,0.9361
mobileNet_model,0.9229,0.9238,0.9229,0.9230


In [73]:
model=tf.keras.models.load_model("/kaggle/input/models/shimaagomaa88/efficentnet/keras/default/1/efficentNetB4_best_weight.keras")


In [72]:
classes = [
    'Calculus',
    'Caries',
    'Gingivitis',
    'Ulcers',
    'Tooth Dicoloration',
    'Hypodontia'
]

In [75]:
def predict_image(image):
    image=image.resize((256,256))
    image=np.array(image)
    image=image.astype('float32')
    image=np.expand_dims(image,axis=0)
    predictions=model.predict(image,verbose=0)
    results={
        classes[i]:float(predictions[0][i])
        for i in ragne(len(classes))
    }
    return results

In [2]:
demo = gr.Interface(
    fn=predict_image,

    inputs=gr.Image(
        type="pil",
        label="Upload Dental Image"
    ),

    outputs=gr.Label(
        num_top_classes=6,
        label="Prediction Results"
    ),

    title="Dental Disease Classification",

    description="Upload a dental image to predict its class."
)

NameError: name 'gr' is not defined

In [1]:
demo.launch(share=True)

NameError: name 'demo' is not defined